In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
import matplotlib.ticker as ticker
from matplotlib.ticker import MaxNLocator

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "Snapshots")
dataType = "Snapshots"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24" 

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetMean(variableSubset, mode="xy"):
    #(1/A) ∫ φ dA
    # dA = R² cos(lat) dlat dlon → weight = cos(lat)

    weights = np.cos(np.deg2rad(variableSubset.latitude))

    if mode == "xy":
        dims = ("latitude", "longitude")
    elif mode == "y":
        dims = ("latitude",)
    elif mode == "x":
        dims = ("longitude",)
    else:
        raise ValueError("mode must be 'xy' or 'y'")

    variableMean = variableSubset.weighted(weights).mean(
        dim=dims,
        skipna=True
    )

    return variableMean

def MeanDBZ(variableSubset, mode="xy"):
    # Convert dBZ → linear Z
    variableSubset_power = 10 ** (variableSubset / 10.0)

    # Take weighted mean
    variableMean = GetMean(variableSubset_power, mode=mode)

    # Convert back to dBZ
    variableMean = 10.0 * np.log10(variableMean)

    return variableMean

In [ ]:
####################################
#LOADING FUNCTIONS

In [ ]:
def GetData(ModelData, varName, t, zTarget=None, multiplyQByRho=True):
        
    #Loading Data
    [dataSubset,dataSubset_diag,dataSubset_static, lat,lon,zGrid_f,zGrid_c, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)
    [zTarget_f, zTarget_c] = ModelData.GetZTarget(zGrid_f, zGrid_c)
        
    #Subsetting Data
    variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
    
    #Interpolating Z levels #*INTERPOLATION
    #################################
    if any(dim.startswith("nVertLevels") for dim in variableSubset.dims):
        if zTarget is None:
            [zTarget_f, zTarget_c] = ModelData.GetZTarget(zGrid_f, zGrid_c)
            zTarget = "loaded"
        variableSubset = ModelData.InterpolateVertical(variableSubset,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
    #################################            

    # mixing ratio (kg/kg) -> mass content (kg/m3)
    if multiplyQByRho and IsMixingRatio(varName):
        rhoSubset = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, "rho")
        rhoSubset = ModelData.InterpolateVertical(rhoSubset, zGrid_f, zGrid_c, zTarget_f, zTarget_c)
        variableSubset = variableSubset * rhoSubset
    
    if varName in ['refl10cm','refl10cm_1km']:
        variableSubset = variableSubset.where(variableSubset > 0)

    return variableSubset

def IsMixingRatio(varName):
    mixingRatios = ["qv", "qc", "qi", "qr", "qs", "qg", "qh"]
    return all(v.strip() in mixingRatios for v in varName.split("+"))

def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    Supports summing multiple variables if varName contains '+' 
    (e.g., 'qc+qi+qg+qs').
    """

    if '+' in varName:
        varList = [v.strip() for v in varName.split('+')]

        variablesum = None
        for var in varList:
            subset = DataOperator_Class.GetData_Variable(
                ModelData,
                dataSubset,
                dataSubset_diag,
                dataSubset_static,
                var
            )

            if variablesum is None:
                variablesum = subset
            else:
                variablesum = variablesum + subset

        variableSubset = variablesum

    else:
        variableSubset = DataOperator_Class.GetData_Variable(
            ModelData,
            dataSubset,
            dataSubset_diag,
            dataSubset_static,
            varName
        )

    return variableSubset

In [ ]:
def GetAllNeededVariables(varName, reflThreshold=15):
    
    # Other Variables
    varData_NSSL = GetData(ModelData_NSSL, varName.replace("'",""),t)
    varData_TEMPO = GetData(ModelData_TEMPO, varName.replace("'",""),t)
    
    zdim = GetVerticalDim(varData_NSSL)
    MeanFunction = GetMean if varName != "refl10cm" else MeanDBZ
    
    if "'" in varName:
        varData_NSSL=varData_NSSL-MeanFunction(varData_NSSL) #for perturbations
        varData_TEMPO=varData_TEMPO-MeanFunction(varData_TEMPO) #for perturbations
    
    #reflectivity threshold
    reflData_NSSL = GetData(ModelData_NSSL, "refl10cm",t)
    reflData_TEMPO = GetData(ModelData_TEMPO, "refl10cm",t)
    if "nVertLevelsP1" in varData_NSSL.coords:
        reflData_NSSL=reflData_NSSL.interp(nVertLevels=varData_NSSL.nVertLevelsP1)
        reflData_TEMPO=reflData_TEMPO.interp(nVertLevels=varData_NSSL.nVertLevelsP1)
    reflData_NSSL = reflData_NSSL.sel({zdim: 3000}, method='nearest') #we only want to see above the storm objects at 3km
    reflData_TEMPO = reflData_TEMPO.sel({zdim: 3000}, method='nearest')
    varData_NSSL_threshold = varData_NSSL.where(reflData_NSSL>reflThreshold)
    varData_TEMPO_threshold = varData_TEMPO.where(reflData_TEMPO>reflThreshold)
    
    subsetData_NSSL = varData_NSSL_threshold.sel({
        zdim: slice(0,16000),
        "longitude": slice(lon_min, lon_max),
        "latitude": slice(lat_min, lat_max)
    })    
    subsetData_TEMPO = varData_TEMPO_threshold.sel({
        zdim: slice(0,16000),
        "longitude": slice(lon_min, lon_max),
        "latitude": slice(lat_min, lat_max)
    })
    
    #mean
    varMean_NSSL = MeanFunction(subsetData_NSSL,mode="y")
    varMean_TEMPO = MeanFunction(subsetData_TEMPO,mode="y")

    varProfile_NSSL = MeanFunction(subsetData_NSSL,mode="xy")
    varProfile_TEMPO = MeanFunction(subsetData_TEMPO,mode="xy")

    return varData_NSSL,varData_TEMPO, subsetData_NSSL,subsetData_TEMPO, varMean_NSSL,varMean_TEMPO, varProfile_NSSL,varProfile_TEMPO

In [ ]:
def LoadOrSave(varName):

    filePath = DataOperator_Class.GetOutputFilePath(
        ModelData_NSSL,
        DirectoryManager,
        outputDirectory,
        fileName=f"outputDictionary_{varName}.nc"
    )

    # ---------------------------------
    # LOAD
    # ---------------------------------
    if os.path.exists(filePath):
        print(f"Loading {filePath}")

        ds = xr.open_dataset(filePath)

        varData_NSSL  = ds["varData_NSSL"]
        varData_TEMPO = ds["varData_TEMPO"]
        subsetData_NSSL  = ds["subsetData_NSSL"]
        subsetData_TEMPO = ds["subsetData_TEMPO"]
        varMean_NSSL     = ds["varMean_NSSL"]
        varMean_TEMPO    = ds["varMean_TEMPO"]
        varProfile_NSSL  = ds["varProfile_NSSL"]
        varProfile_TEMPO = ds["varProfile_TEMPO"]

    # ---------------------------------
    # RUN + SAVE
    # ---------------------------------
    else:
        print(f"Running GetAllNeededVariables for {varName}")

        varData_NSSL, varData_TEMPO, \
        subsetData_NSSL, subsetData_TEMPO, \
        varMean_NSSL, varMean_TEMPO, \
        varProfile_NSSL, varProfile_TEMPO = GetAllNeededVariables(varName=varName)

        os.makedirs(os.path.dirname(filePath), exist_ok=True)

        ds = xr.Dataset({
            "varData_NSSL": varData_NSSL,
            "varData_TEMPO": varData_TEMPO,
            "subsetData_NSSL": subsetData_NSSL,
            "subsetData_TEMPO": subsetData_TEMPO,
            "varMean_NSSL": varMean_NSSL,
            "varMean_TEMPO": varMean_TEMPO,
            "varProfile_NSSL": varProfile_NSSL,
            "varProfile_TEMPO": varProfile_TEMPO,
        })

        ds.to_netcdf(filePath)
        print(f"Saved to {filePath}")

    return (
        varData_NSSL,
        varData_TEMPO,
        subsetData_NSSL,
        subsetData_TEMPO,
        varMean_NSSL,
        varMean_TEMPO,
        varProfile_NSSL,
        varProfile_TEMPO
    )

In [ ]:
####################################
#LOADING DICTIONARIES

In [ ]:
timeStringDictionary = {
    "TRACER": {
        "WET": {
            0: f"2022-07-01_18.00.00",
        },
        "DIURNAL": {
            -5: f"2022-06-22_21.00.00",
        },
    },
    "PRECIP": {
        "WET": {
            12: f"2022-06-07_03.00.00",
        },
        "DIURNAL": {
            12: f"2022-07-16_05.00.00",
        },
    },
    "Hawaii": {
        "WET": {
            12: f"2021-12-06_23.00.00",
        },
        "TRADES": {
            24: f"2022-08-08_00.45.00",#f"2022-08-08_01.00.00",
        },
    },
}

selectedTimeString = timeStringDictionary[ModelData_NSSL.region][ModelData_NSSL.case][int(ModelData_NSSL.spinup_hours)]
t = ModelData_NSSL.timeStrings.index(selectedTimeString)

In [ ]:
rectangleBoundsDictionary = {
    "TRACER": {
        "WET": {
            0: [-94.5,-92, 28.5,29.9], #[lon_min,lon_max, lat_min,lat_max]
        },
        "DIURNAL": {
            -5: [-96,-92, 29.85,32.75],
        },
    },
    "PRECIP": {
        "WET": {
            12: [119.5,121.25, 22.25,24.25],
        },
        "DIURNAL": {
            12: [120.2,122.25, 22.4,25],
        },
    },
    "Hawaii": {
        "WET": {
            12: [-158,-156.75, 18.75,22],
        },
        "TRADES": {
            24: [-160.5,-159.25, 21.5,22.5],
        },
    },
}

[lon_min,lon_max, lat_min,lat_max] = rectangleBoundsDictionary[ModelData_NSSL.region][ModelData_NSSL.case][int(ModelData_NSSL.spinup_hours)]

def GetAverageMode(region, case):
    # "y": average over latitude -> x axis is longitude
    # "x": average over longitude -> x axis is latitude
    return "x" if (region == "Hawaii" and case == "WET") else "y"

avgMode = GetAverageMode(ModelData_NSSL.region, ModelData_NSSL.case)
xAxisName = "latitude" if avgMode == "x" else "longitude"

In [ ]:
#Setting up Color Ticks and CMaps
climLevelsDictionary = {"refl10cm": np.linspace(15,60,50),
                        "qc+qi+qg": np.linspace(0,1,50),
                        "qs": np.linspace(0,1,50),
                        "w": np.linspace(-2,2,50),
                        "qr": np.linspace(0,1,50),
                        "relhum": np.linspace(50,105,50)
                       }

if (Region,Case)==('Hawaii', 'WET'):
    climLevelsDictionary["qc+qi+qg"] = np.linspace(0,1.75,50)
    climLevelsDictionary["qr"] = np.linspace(0,1.75,50)

cmapDictionary = {"refl10cm": "turbo",
                  "qc+qi+qg": "turbo",
                  "qs": "turbo",
                  "w": "RdBu_r",
                  "qr": "turbo",
                  "relhum": "turbo"
                 }

def GenerateTicksAndLim(data, nTicks=6, nLevels=15):
    if isinstance(data, (list, tuple)) and len(data) > 0 and not np.isscalar(data[0]):
        data = np.concatenate([np.asarray(d).ravel() for d in data])
    else:
        data = np.asarray(data)
    
    dMin, dMax = np.nanmin(data), np.nanmax(data)
    locator = MaxNLocator(nbins=nTicks - 1)
    ticks = locator.tick_values(dMin, dMax)
    lim = (ticks[0], ticks[-1])
    
    # Snap nLevels to the nearest value that's an exact multiple of the
    # actual tick spacing, so every tick lands precisely on a level edge
    # instead of only the first/last.
    nTickGaps = len(ticks) - 1
    subdivisionsPerTick = max(1, round((nLevels - 1) / nTickGaps))
    nLevelsActual = nTickGaps * subdivisionsPerTick + 1
    
    levels = np.linspace(lim[0], lim[1], nLevelsActual)
    return [ticks, lim, levels]

def GetTicksDictionary(climLevelsDictionary):
    fixedLevelsTicks = {
        "refl10cm": (np.linspace(15, 60, 91), np.arange(15, 61, 5)),   # 0.5 spacing, ticks 15,20,...,60
    }
    if (Region, Case) == ('Hawaii', 'WET'):
        for name in ["qc+qi+qg", "qr"]:
            fixedLevelsTicks[name] = (np.linspace(0, 1.75, 71), np.arange(0, 1.76, 0.25))   # 0.025 spacing, ticks every 0.25

    ticksDictionary = {}
    for name in list(climLevelsDictionary):
        if name in fixedLevelsTicks:
            levels, ticks = fixedLevelsTicks[name]
        else:
            lv = climLevelsDictionary[name]
            ticks, lim, levels = GenerateTicksAndLim([lv[0], lv[-1]], nLevels=len(lv))
        climLevelsDictionary[name] = levels
        ticksDictionary[name] = ticks
    return [ticksDictionary]
    
[ticksDictionary]=GetTicksDictionary(climLevelsDictionary)

In [ ]:
def MakeVariablesDictionary(varName,
                            varData_NSSL, varData_TEMPO,
                            subsetData_NSSL, subsetData_TEMPO,
                            varMean_NSSL, varMean_TEMPO,
                            varProfile_NSSL, varProfile_TEMPO,
                            variablesDictionary=None):

    if variablesDictionary is None:
        variablesDictionary = {}
    variablesDictionary[varName] = {
        "varData_NSSL": varData_NSSL,
        "varData_TEMPO": varData_TEMPO,
        "subsetData_NSSL": subsetData_NSSL,
        "subsetData_TEMPO": subsetData_TEMPO,
        "varMean_NSSL": varMean_NSSL,
        "varMean_TEMPO": varMean_TEMPO,
        "varProfile_NSSL": varProfile_NSSL,
        "varProfile_TEMPO": varProfile_TEMPO
    }

    return variablesDictionary




In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def PlotRectangle(ax,
                  lon_min, lon_max,
                  lat_min, lat_max):
    
    rect = Rectangle(
        (lon_min, lat_min),
        lon_max - lon_min,
        lat_max - lat_min,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )
    
    ax.add_patch(rect)
from matplotlib.patches import Rectangle

In [ ]:
# def PlotRectangleFromCenterLine(ax,
#                                 lon0, lat0,
#                                 lon1, lat1,
#                                 width,
#                                 edgecolor='red',
#                                 linewidth=2):

#     # direction vector
#     dx = lon1 - lon0
#     dy = lat1 - lat0
#     L = np.sqrt(dx**2 + dy**2)

#     ux = dx / L
#     uy = dy / L

#     # perpendicular vector
#     px = -uy
#     py = ux

#     # half width
#     w = width / 2

#     # rectangle corners
#     p1 = (lon0 + px*w, lat0 + py*w)
#     p2 = (lon1 + px*w, lat1 + py*w)
#     p3 = (lon1 - px*w, lat1 - py*w)
#     p4 = (lon0 - px*w, lat0 - py*w)

#     rect = Polygon([p1, p2, p3, p4],
#                    closed=True,
#                    edgecolor=edgecolor,
#                    facecolor='none',
#                    linewidth=linewidth)

#     ax.add_patch(rect)
# from matplotlib.patches import Polygon

In [ ]:
def GetVerticalDim(data):
    if "nVertLevels" in data.dims:
        return "nVertLevels"
    elif "nVertLevelsP1" in data.dims:
        return "nVertLevelsP1"
    else:
        raise ValueError("No vertical dimension found")

In [ ]:
def PlotHorizontal(subsetData, varName,
                   ax=None, cmap=None, levels=None):

    scaleFactor = 1000 if IsMixingRatio(varName) else 1
    subsetData_plot = subsetData * scaleFactor

    if ax is None:
        fig, ax = plt.subplots()

    zdim = GetVerticalDim(subsetData)

    subsetData_plot = subsetData_plot.sel({
        zdim: slice(0,16000),
        "longitude": slice(lon_min, lon_max),
        "latitude": slice(lat_min, lat_max)
    })

    if cmap is None:
        cmap = 'turbo'

    p = subsetData_plot.sel({zdim:3000}, method='nearest').plot(
        ax=ax,
        cmap=cmap,
        levels=levels,
        add_colorbar=False,
        add_labels=False,
    )

    ax.set_title(GetPlotLabel(varName), loc='center', fontsize=12, pad=3)

    return p

def GetPlotLabel(varName):
    """ρ_d·q label (g m^-3) for mixing-ratio variables; other variables unchanged."""
    if not IsMixingRatio(varName):
        return varName
    terms = [v.strip() for v in varName.split("+")]
    q = "+".join(f"q_{{{v[1:]}}}" for v in terms)   # "qc" -> q_{c}
    if len(terms) > 1:
        q = f"({q})"
    return rf"$\rho_d{q}$"

def SetCbarTicks(cb, varName):
    if varName in ticksDictionary:
        cb.set_ticks(ticksDictionary[varName])
    return cb

def PlotVertical(varMean, varName,
                 ax=None, cmap=None, levels=None):

    scaleFactor = 1000 if IsMixingRatio(varName) else 1
    varMean_plot = varMean * scaleFactor

    zdim = GetVerticalDim(varMean)
    varMean_plot = varMean_plot.assign_coords({zdim: varMean_plot[zdim] / 1000})
    if ax is None:
        fig, ax = plt.subplots()

    varMean_plot = varMean_plot.sel({
        "longitude": slice(lon_min, lon_max)
    })

    if cmap is None:
        cmap = 'turbo'

    p = varMean_plot.plot(
        ax=ax,
        cmap=cmap,
        levels=levels,
        add_colorbar=False,
    )
    ax.set_ylim(0,16);
    ax.set_yticks(np.arange(0, 17, 2))
    ax.set_title(GetPlotLabel(varName), loc='center', fontsize=12, pad=3)

    return p

def PlotHorizontalVertical_V1(subsetData, varMean, varName,
                           cmap='turbo', levels=None):

    fig = plt.figure(figsize=(12,4))

    outer = gridspec.GridSpec(1, 2, width_ratios=[20,1], wspace=0.05)

    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[0], wspace=0.25
    )

    ax0 = fig.add_subplot(inner[0])
    ax1 = fig.add_subplot(inner[1])
    cax = fig.add_subplot(outer[1])

    p = PlotHorizontal(subsetData, varName, ax0, cmap, levels)
    PlotVertical(varMean, varName, ax1, cmap, levels)

    cb=fig.colorbar(p, cax=cax, label=GetPlotLabel(varName))
    SetCbarTicks(cb, varName)

    ax0.set_title("Horizontal Zoom-In")
    ax1.set_title("Average")

    return fig
# #EXAMPLE
# mpType = "NSSL"
# fig = PlotHorizontalVertical_V1(variablesDictionary[varName][f"subsetData_{mpType}"],variablesDictionary[varName][f"varMean_{mpType}"],varName=varName,
#                        cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
# SaveFigure(ModelData_NSSL, fig, plotType=f"HorizontalVertical_NSSL_{varName}")

def PlotHorizontalVertical_V2(subsetData_NSSL, varMean_NSSL, 
                               subsetData_TEMPO, varMean_TEMPO, 
                               varName, cmap='turbo', levels=None):

    # Increased figure height to 8 to accommodate the second row
    fig = plt.figure(figsize=(12, 8))

    # Outer grid now has 2 rows, 2 columns (Plots column and Colorbar column)
    outer = gridspec.GridSpec(2, 2, width_ratios=[20, 1], wspace=0.05, hspace=0.3)

    # --- ROW 1: NSSL ---
    inner_nssl = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[0, 0], wspace=0.25
    )
    ax0_nssl = fig.add_subplot(inner_nssl[0])
    ax1_nssl = fig.add_subplot(inner_nssl[1])

    # --- ROW 2: TEMPO ---
    inner_tempo = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[1, 0], wspace=0.25
    )
    ax0_tempo = fig.add_subplot(inner_tempo[0])
    ax1_tempo = fig.add_subplot(inner_tempo[1])

    # Shared Colorbar Axis (Spans both rows on the right)
    cax = fig.add_subplot(outer[:, 1])

    # Plot NSSL Row
    p = PlotHorizontal(subsetData_NSSL, varName, ax0_nssl, cmap, levels)
    PlotVertical(varMean_NSSL, varName, ax1_nssl, cmap, levels)
    
    # Plot TEMPO Row
    PlotHorizontal(subsetData_TEMPO, varName, ax0_tempo, cmap, levels)
    PlotVertical(varMean_TEMPO, varName, ax1_tempo, cmap, levels)

    # Add the shared colorbar
    cb = fig.colorbar(p, cax=cax, label=GetPlotLabel(varName))
    SetCbarTicks(cb, varName)
    
    # Apply your 1-decimal formatting fix
    import matplotlib.ticker as ticker
    cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

    # Setting Titles
    ax0_nssl.set_title("NSSL: Horizontal Zoom-In")
    ax1_nssl.set_title("NSSL: Average")
    ax0_tempo.set_title("TEMPO: Horizontal Zoom-In")
    ax1_tempo.set_title("TEMPO: Average")

    return fig

def PlotFullReflectivity(mpType,
                         ax=None):
    
    varName = "refl10cm"
    
    #Make Plot
    if ax is None:
        fig, ax = plt.subplots()
    p = variablesDictionary[varName][f"varData_{mpType}"].sel(nVertLevels=3000, method='nearest').plot(
        ax=ax, cmap='turbo', add_labels=False, add_colorbar=False,
        levels=climLevelsDictionary[varName],     # 15-50, same levels as the cross-sections
    )
    # p.colorbar.set_label('refl10cm')
    ax.set_title(varName, loc='center', fontsize=12, pad=3)
    
    #Draw Rectangle
    PlotRectangle(ax,
                  lon_min, lon_max,
                  lat_min, lat_max)

    return p

In [ ]:
def PlotVerticalProfiles(varProfile_NSSL, varProfile_TEMPO, varName, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    
    # 1. Scaling logic
    scaleFactor = 1000 if IsMixingRatio(varName) else 1
    
    # 2. Get vertical dimension and convert meters to km
    zdim = GetVerticalDim(varProfile_NSSL)
    
    # Create plotting data with km coordinates
    # We do this once to keep the code efficient
    vn_plot = (varProfile_NSSL * scaleFactor).assign_coords({zdim: varProfile_NSSL[zdim] / 1000})
    vt_plot = (varProfile_TEMPO * scaleFactor).assign_coords({zdim: varProfile_TEMPO[zdim] / 1000})

    # 3. Plotting lines
    ax.plot(vn_plot, vn_plot[zdim], color='blue', label='NSSL', linewidth=2)
    ax.plot(vt_plot, vt_plot[zdim], color='green', label='TEMPO', linewidth=2)

    # 4. Range and Ticks
    ax.set_ylim(0, 16)
    ax.set_yticks(np.arange(0, 17, 2))
    
    # 5. FONT SETTINGS (The Fix)
    # Ticks (the numbers)
    ax.tick_params(axis='both', labelsize=fontSettings["tickFont"]+4)
    if varName == "w":
        from matplotlib.ticker import MultipleLocator
        ax.xaxis.set_major_locator(MultipleLocator(0.5))

    # Axis Labels (the text)
    ax.set_xlabel(GetPlotLabel(varName), fontsize=fontSettings["labelFont"]+10)
    ax.set_ylabel("Altitude (km)", fontsize=fontSettings["labelFont"]+10)
    
    # Grid
    ax.grid(True, linestyle='--', alpha=0.7)

    ax.set_xlim(profileLimsLevelsDictionary[varName][0],profileLimsLevelsDictionary[varName][-1])
    
    return ax

profileLimsLevelsDictionary = {"refl10cm": np.linspace(0,50,50),
                        "qc+qi+qg": np.linspace(0,1,50),
                        "qs": np.linspace(0,1,50),
                        "w": np.linspace(-0.75,1,50),
                        "qr": np.linspace(0,0.6,50),
                        "relhum": np.linspace(35,105,50)
                       }

def PlotAllVerticalProfiles(variablesDictionary, varNames, ModelData_NSSL, fontSettings):
    
    # 1. Setup the figure and GridSpec
    fig = plt.figure(figsize=(22, 4))
    gs = gridspec.GridSpec(1, len(varNames), figure=fig, wspace=0.15)

    # Create axes
    axs = [fig.add_subplot(gs[i]) for i in range(len(varNames))]

    # 2. Loop through variables
    for i, (ax, varName) in enumerate(zip(axs, varNames)):

        PlotVerticalProfiles(
            variablesDictionary[varName]["varProfile_NSSL"],
            variablesDictionary[varName]["varProfile_TEMPO"],
            varName,
            ax
        )

        # 3. Strip y-axis for all but first subplot
        if i > 0:
            ax.set_ylabel("")
            ax.set_yticklabels([])
            ax.tick_params(
                axis='y',
                which='both',
                left=False,
                right=False
            )

    # 4. Custom Legend in middle subplot
    legendIndex = len(axs) // 2

    customLines = [
        Line2D([0], [0], color='blue', lw=2),
        Line2D([0], [0], color='green', lw=2)
    ]

    axs[legendIndex].legend(
        customLines,
        ['NSSL', 'TEMPO'],
        loc='upper right',
        fontsize=fontSettings["tickFont"],
        frameon=True
    )

    # 5. Main Title
    fig.suptitle(
        f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
        fontsize=fontSettings["titleFont"]+6,
        y=1.01,
        fontweight="bold"
    )
    
    return fig, axs

In [ ]:
def GetOutputFile(ModelData, outputPlottingDirectory):
    outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{ModelData.spinup_hours}hrs"
    
    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory)
    os.makedirs(outputFilePath, exist_ok=True)
    return outputFilePath


def SaveFigure(ModelData, fig, plotType):
    """
    Saves a figure to corresponding directory.
    """
    # --- Define output subdirectory and file path ---
    outputFilePath = GetOutputFile(ModelData, outputPlottingDirectory)
    outputFile = os.path.join(outputFilePath,f"{plotType}.png")

    # --- Save figure ---
    fig.savefig(outputFile, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
####################################
#PLOTTING
fontSettings = {
    "tickFont": 16,
    "labelFont": 10,
    "legendFont": 14,
    "titleFont": 22+4,
}

In [ ]:
#Loading Data
varNames = ["refl10cm", 
            "qc+qi+qg","qs","w",
            "qr","relhum"]
variablesDictionary = {}
for varName in tqdm(varNames):
    
    [varData_NSSL,varData_TEMPO, subsetData_NSSL,subsetData_TEMPO, varMean_NSSL,varMean_TEMPO, varProfile_NSSL,varProfile_TEMPO] = LoadOrSave(varName=varName) #GetAllNeededVariables
    variablesDictionary = MakeVariablesDictionary(varName,
                                                  varData_NSSL, varData_TEMPO,
                                                  subsetData_NSSL, subsetData_TEMPO,
                                                  varMean_NSSL, varMean_TEMPO,
                                                  varProfile_NSSL, varProfile_TEMPO,
                                                  variablesDictionary)

In [ ]:
#Horizontal Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 8))

# --- Plot 1: NSSL ---
mpType = "NSSL"
p1 = PlotFullReflectivity(mpType, ax1)
cb1 = fig.colorbar(p1, ax=ax1, pad=0.02)

# --- Plot 2: TEMPO ---
mpType = "TEMPO"
p2 = PlotFullReflectivity(mpType, ax2)
# Add colorbar
cb2 = SetCbarTicks(fig.colorbar(p2, ax=ax2, pad=0.02),varName='refl10cm')

SaveFigure(ModelData_NSSL, fig, plotType="ReflectivityBoundingBox")

In [ ]:
#Plotting Indiviudal Vertical Cross-Sections
varNames = ["refl10cm", "qc+qi+qg","qs","w","qr","relhum"]
for varName in varNames:
    fig = PlotHorizontalVertical_V2(variablesDictionary[varName][f"subsetData_NSSL"],variablesDictionary[varName][f"varMean_NSSL"],
                                    variablesDictionary[varName][f"subsetData_TEMPO"],variablesDictionary[varName][f"varMean_TEMPO"],
                                    varName=varName,
                                    cmap=cmapDictionary[varName],levels=climLevelsDictionary[varName])
    SaveFigure(ModelData_NSSL, fig, plotType=f"HorizontalVertical_{varName}")

In [ ]:
#Plotting Individual Profiles
fig, axs = PlotAllVerticalProfiles(
    variablesDictionary,
    varNames,
    ModelData_NSSL,
    fontSettings
)

SaveFigure(ModelData_NSSL, fig, plotType="VerticalProfiles")
fig

In [ ]:
############################################################################
""" LARGE SUBPLOT """

In [ ]:
def FormatAllColorbars(fig, integer_axes_list, default_fmt='%.1f', int_fmt='%.0f'):
    """
    Standardizes all colorbars in a figure.
    - integer_axes_list: List of axes whose colorbars should have 0 decimals.
    - default_fmt: Format for all other colorbars (default 1 decimal).
    - int_fmt: Format for specified axes (default 0 decimals).
    """
    for ax in fig.get_axes():
        if ax.get_label() == '<colorbar>':
            # Check if the colorbar belongs to one of our special axes
            try:
                parent_ax = ax._colorbar_info['parents'][0]
                
                if parent_ax in integer_axes_list:
                    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(int_fmt))
                else:
                    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(default_fmt))
                
                ax.tick_params(labelsize=fontSettings["labelFont"])
            except (AttributeError, IndexError):
                # Fallback if the internal matplotlib structure differs
                continue
import matplotlib.ticker as ticker

In [ ]:
def CombinedPlot():
    
    #Setting Up Figure
    fig = plt.figure(figsize=(12,6))
    gs = gridspec.GridSpec(3, 4, figure=fig, wspace=0.12, hspace=0.15)
    axs = gs.subplots()
    
    # varName: (row, NSSL column). TEMPO uses the mirrored column (3 - column).
    layout = {
        "refl10cm": (0, 0),
        "qc+qi+qg": (0, 1),
        "w":        (1, 0),
        "qr":       (2, 0),
        "qs":       (2, 1),
    }
    mapCol = 1   # reflectivity map: NSSL at (1,1), TEMPO at (1,2)
    
    for mpType in ["NSSL", "TEMPO"]:
        colFor = (lambda c: 3 - c) if mpType == "TEMPO" else (lambda c: c)
    
        # Reflectivity map (no preset levels, so no SetCbarTicks)
        ax = axs[1, colFor(mapCol)]
        p = PlotFullReflectivity(mpType, ax)
        SetCbarTicks(fig.colorbar(p, ax=ax, pad=0), "refl10cm")
    
        # Vertical cross-sections
        for varName, (row, col) in layout.items():
            ax = axs[row, colFor(col)]
            p = PlotVertical(variablesDictionary[varName][f"varMean_{mpType}"], varName,
                             ax=ax, cmap=cmapDictionary[varName], levels=climLevelsDictionary[varName])
            SetCbarTicks(fig.colorbar(p, ax=ax, pad=0), varName)
    
    #Fixing up ticks and labels
    for ax in axs[:, 1:].flat:            # y labels only on the left column
        ax.tick_params(left=False, labelleft=False)
        ax.set_ylabel("")
    for ax in axs[:-1, :].flat:           # x labels only on the bottom row
        ax.tick_params(bottom=False, labelbottom=False)
        ax.set_xlabel("")
    for ax in axs[-1, :].flat:
        ax.set_xlabel(xAxisName)          # "longitude" or "latitude" (defined with avgMode); use "longitude" if you skipped that step
    for ax in axs[:, 0]:
        ax.set_ylabel("Altitude (km)")
    
    FormatAllColorbars(fig, [axs[1, 1], axs[1, 2]])

    return [fig]

In [ ]:
[fig]=CombinedPlot()
SaveFigure(ModelData_NSSL, fig, plotType="CombinedSubplot")
fig